In [1]:
import pyfastx

In [2]:
DATA_PATH = r'download.20260314.151350\Phytozome\PhytozomeV12\early_release\Ptrichocarpa_444_v3.1\assembly\Ptrichocarpa_444_v3.0.fa.gz'

data = pyfastx.Fasta(DATA_PATH)

print(f"Number of sequences : {len(data):,}")
print(f"Total genome size   : {data.size:,} bp")
print(f"GC content          : {data.gc_content:.2f}%")

for i, seq in enumerate(data):
    print(f"  {seq.name:<30}  length={len(seq):>12,} bp")


Number of sequences : 1,446
Total genome size   : 434,134,862 bp
GC content          : 33.75%
  Chr01                           length=  50,495,391 bp
  Chr02                           length=  25,263,035 bp
  Chr03                           length=  21,816,808 bp
  Chr04                           length=  24,267,051 bp
  Chr05                           length=  25,890,704 bp
  Chr06                           length=  27,912,125 bp
  Chr07                           length=  15,610,913 bp
  Chr08                           length=  19,465,461 bp
  Chr09                           length=  12,948,742 bp
  Chr10                           length=  22,580,532 bp
  Chr11                           length=  18,501,271 bp
  Chr12                           length=  15,760,346 bp
  Chr13                           length=  16,320,717 bp
  Chr14                           length=  18,920,894 bp
  Chr15                           length=  15,278,577 bp
  Chr16                           length=  14,494,3

## I) Charactere Level

In [ ]:
dna_voc = {  
    'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4
}

id2base = {v: k for k, v in dna_voc.items()} 

all_token_ids = {}

for seq in data:
    
    #Encode 'ACGTN' => [0, 1, 2, 3, 4]
    ids = [dna_voc.get(b.upper(), 4) for b in seq.seq]
    all_token_ids[seq.name] = ids
 
 
    #Decode for sanity check
    decoded = ''.join(id2base[i] for i in ids)
    
    
    
    print(f"{seq.name:<30}  {len(seq):>12,} bp  →  {len(ids):>12,} tokens")
    print(f"  original : {seq.seq[:20]}")
    print(f"  encoded  : {ids[:20]}")
    print(f"  decoded  : {decoded[:20]}")
    print()

print(f"Done — {len(all_token_ids)} sequences tokenized")

## II) K-mer

In [ ]:
from itertools import product

k = 6
kmers = [''.join(p) for p in product('ACGT', repeat=k)] 

kmer_voc = {kmer: i for i, kmer in enumerate(kmers)}

all_token_ids = {}

for seq in data:
    
    dna_string = seq.seq.upper()
    
    ids = [
        kmer_voc.get(dna_string[i:i+k], -1)  for i in range(0, len(dna_string) - k + 1, k)
    ]
    
    all_token_ids[seq.name] = ids
    
    print(f"{seq.name:<30}  {len(seq):>12,} bp  →  {len(ids):>12,} tokens")
    print(f" first 3 k-mers: {[dna_string[i:i+k] for i in range(0, 3*k, k)]}")
    print(f" encoded: {ids[:3]}")
    print()

print(f"Done - {len(all_token_ids)} sequences tokenized")

## III) Overlaping k-mer tokenizer (stride = 1)

In [ ]:
k = 6
stride = 1

kmers   = [''.join(p) for p in product('ACGT', repeat=k)]
kmer_voc = {kmer: i for i, kmer in enumerate(kmers)}

print(f"Vocab size: {len(kmer_voc):,} possible {k}-mers")

all_token_ids = {}

for seq in data:

    dna_string = seq.seq.upper()

    ids = [
        kmer_voc.get(dna_string[i:i+k], -1) for i in range(0, len(dna_string) - k + 1, stride)
        ]

    all_token_ids[seq.name] = ids
    
    print(f"{seq.name:<30}  {len(seq):>12,} bp  →  {len(ids):>12,} tokens")
    print(f"first 4 k-mers : {[dna_string[i:i+k] for i in range(0, 4*stride, stride)]}")
    print(f"encoded : {ids[:4]}")
    print()

print(f"Done - {len(all_token_ids)} sequences tokenized")
    
    

## IV) BPE tokenisation

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Split

tokenizer = Tokenizer(BPE(unk_token='[UNK]'))


trainer = BpeTrainer(
    vocab_size=4096,
    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    min_frequency=2,
    show_progress=True,
)

tokenizer.train(['dna_corpus.txt'], trainer)
tokenizer.save('dna_bpe.json')


## V) DNABERT-2 tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True 
)

In [ ]:
data_dna = [str(seq.seq).upper() for seq in data]

In [ ]:
import torch

encoded = tokenizer(
        data_dna,
        max_length = 4096,
        padding = "max_length",
        truncation = True,
        return_tensors = "pt",
        return_attention_mask = True,
    )

#torch.save(encoded, "BERT2_encoded.pt")